# 32. v2 + XGB 시드 10개 평균 (최종 제출 후보)

## 배경 (11/13)
- v2 = C(22b XGB 0.7 + LR 0.3) + XGB 시드 5개 평균 + **test 기반 규칙 제거** (`load_data(dedupe=False)`, `ratio`에 in-frame indel 포함) → **LB 0.3792 (팀 최고)**
- 모델·피처·섞기 비율은 v2 그대로 두고 **XGB 시드만 5개 → 10개**로 늘려 예측을 안정시킴 (새 피처·새 모델 없음)
- 시드 평균은 C에서 이미 LB +0.001 (0.3650 → 0.3660)으로 효과 확인

## 검증 방법
- 시드 10개 = 기존 5개(42, 7, 123, 2024, 777) + 새 5개(1, 99, 314, 1004, 2026)
  - 앞 5개만 평균하면 v2가 그대로 재현되어야 함 (CV 0.4641 / OOF 0.4651) → 재현 확인 후 10개와 비교
- 비교: 같은 fold에서 v2(시드 5개) vs 시드 10개
- 참고 셀: 섞기 비율 0.6 / 0.7 / 0.8을 **train OOF로만** 확인 (LB를 보고 비율을 고르지 않음)
- 실행 환경: Windows 데스크탑, 제출 파일도 Windows에서만 저장 (팀 규칙)
- 예상 시간: 22b 1회 약 160초 × 10 ≈ **27분** (시드마다 중간 저장 → 끊겨도 이어서 실행 가능)

#### 준비: 데이터, 피처, 10 최적 설정 (22번과 같음, `dedupe=False`)

In [1]:
import sys; sys.path.append('..')
import platform, time, glob, numpy as np, pandas as pd, xgboost as xgb
from pathlib import Path
from collections import Counter
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from src.config import SEED, WT
from src.data import load_data, get_gene_cols
from src.features import mutation_features, aa_substitution, mutation_type
from src.validation import seed_everything, encode_target, get_folds, run_cv, save_submission, macro_f1

print(f'{platform.system()} | CPU {platform.machine()} | xgboost {xgb.__version__}')
seed_everything()
train, test, submission = load_data(dedupe=False)          # v2: 이소폼 중복 제거 끔
genes = get_gene_cols(train)
y, le = encode_target(train)
folds = get_folds(y)
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)
assert 'r_inframe_indel' in F_tr['ratio'].columns, 'features.py가 v2(indel 포함)가 아님 → git pull 확인'

class BalancedXGB(xgb.XGBClassifier):
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

BEST = dict(max_depth=3, learning_rate=0.1, n_estimators=200, colsample_bytree=0.3,
            tree_method='hist', eval_metric='mlogloss', random_state=SEED, n_jobs=-1)
print('ratio 열:', list(F_tr['ratio'].columns))

Windows | CPU AMD64 | xgboost 3.4.1
ratio 열: ['r_silent', 'r_missense', 'r_nonsense', 'r_frameshift', 'r_inframe_indel', 'r_stop_loss', 'r_other', 'n_mut_log']


#### 22b 피처: any + ratio + 미스센스 치환 종류 비율 (상위 100개는 fold 학습 부분에서만 선택)

In [2]:
def sub_ratio_matrix(df):
    # 환자별 미스센스 치환 종류 비율 (한 환자 안에서만 계산 → 누수 아님)
    rows = []
    for row in df[genes].to_numpy():
        subs = (aa_substitution(t) for v in row if v != WT for t in v.split() if mutation_type(t) == 'missense')
        c = Counter(s for s in subs if s is not None)
        tot = sum(c.values())
        rows.append({k: n / tot for k, n in c.items()} if tot else {})
    M = pd.DataFrame(rows, index=df.index).fillna(0).astype(np.float32)
    M.columns = ['sub__' + k.replace('>', '_') for k in M.columns]
    return M

class TopNSubs(BaseEstimator, TransformerMixin):
    # fold 학습 부분에서 비율 합이 큰 상위 n개 치환 열만 남김. 목록은 fit에서만 정함
    def __init__(self, n=30):
        self.n = n
    def fit(self, X, y=None):
        sub = [c for c in X.columns if c.startswith('sub__')]
        keep = X[sub].sum().sort_values(ascending=False).index[:self.n]
        self.drop_ = [c for c in sub if c not in set(keep)]
        self.kept_ = list(keep)
        return self
    def transform(self, X):
        return X.drop(columns=self.drop_)

S_tr = sub_ratio_matrix(train)
S_te = sub_ratio_matrix(test).reindex(columns=S_tr.columns, fill_value=0)   # 열은 train 기준
X22b    = pd.concat([F_tr['any'], F_tr['ratio'], S_tr], axis=1)
X22b_te = pd.concat([F_te['any'], F_te['ratio'], S_te], axis=1)
print('X22b', X22b.shape, '| X22b_te', X22b_te.shape)

X22b (6201, 4633) | X22b_te (2546, 4633)


#### LR (14·C와 같은 설정: any, C=0.1, balanced)

In [3]:
r_lr = run_cv(lambda: LogisticRegression(C=0.1, max_iter=3000, class_weight='balanced', random_state=SEED),
              F_tr['any'].astype('float32'), y, F_te['any'].astype('float32'), folds=folds, verbose=False)
print(f"LR 단독 CV {r_lr['cv_mean']:.4f}")

fs = lambda P: np.array([macro_f1(y[va], P[va].argmax(1)) for _, va in folds])
coad, stes = list(le.classes_).index('COAD'), list(le.classes_).index('STES')
apc_te = test['APC'].map(lambda v: v != WT and any(mutation_type(t) in ('nonsense', 'frameshift') for t in v.split())).to_numpy()

LR 단독 CV 0.3818


#### XGB 시드 10개 (시드마다 중간 저장)
- `../submissions/cache32/seed_<시드>.npz`에 저장 → 중간에 끊기면 이 셀만 다시 실행하면 끝난 시드는 건너뜀

In [ ]:
SEEDS_OLD = [42, 7, 123, 2024, 777]       # v2와 같은 시드
SEEDS_NEW = [1, 99, 314, 1004, 2026]
SEEDS = SEEDS_OLD + SEEDS_NEW
CACHE = Path('../submissions/cache32'); CACHE.mkdir(parents=True, exist_ok=True)

oofs, tes, seed_cv = {}, {}, {}
for s in SEEDS:
    f = CACHE / f'seed_{s}.npz'
    if f.exists():
        d = np.load(f); oofs[s], tes[s] = d['oof'], d['test']
        seed_cv[s] = fs(oofs[s]).mean()
        print(f'seed {s}: 저장본 사용 (22b CV {seed_cv[s]:.4f})')
        continue
    t = time.time()
    p = dict(BEST, random_state=s)
    r = run_cv(lambda: make_pipeline(TopNSubs(100), BalancedXGB(**p)), X22b, y, X22b_te, folds=folds, verbose=False)
    oofs[s], tes[s], seed_cv[s] = r['oof_proba'], r['test_proba'], r['cv_mean']
    np.savez(f, oof=oofs[s], test=tes[s])
    print(f'seed {s}: 22b CV {seed_cv[s]:.4f} ({time.time() - t:.0f}s)')

v = np.array(list(seed_cv.values()))
print(f'\n시드별 22b CV 범위 {v.min():.4f} ~ {v.max():.4f} (차이 {v.max() - v.min():.4f})')

seed 42: 22b CV 0.4541 (184s)
seed 7: 22b CV 0.4531 (174s)
seed 123: 22b CV 0.4475 (179s)
seed 2024: 22b CV 0.4559 (177s)
seed 777: 22b CV 0.4555 (172s)
seed 1: 22b CV 0.4580 (175s)
seed 99: 22b CV 0.4533 (172s)
seed 314: 22b CV 0.4543 (172s)
seed 1004: 22b CV 0.4504 (170s)
seed 2026: 22b CV 0.4558 (179s)

시드별 22b CV 범위 0.4475 ~ 0.4580 (차이 0.0106)


#### v2(시드 5개) 재현 → 시드 10개와 비교
- 시드 5개 결과가 v2 기록(CV 0.4641 / OOF 0.4651)과 같아야 함 → 다르면 데이터·코드 설정이 v2와 다른 것이니 제출하지 말고 확인

In [ ]:
blend = lambda seeds, w=0.7: (w * np.mean([oofs[s] for s in seeds], axis=0) + (1 - w) * r_lr['oof_proba'],
                              w * np.mean([tes[s] for s in seeds], axis=0)  + (1 - w) * r_lr['test_proba'])
V2_oof, V2_te = blend(SEEDS_OLD)
S10_oof, S10_te = blend(SEEDS)
s5, s10 = fs(V2_oof), fs(S10_oof)

rows = []
for name, po, pt in [('v2: 시드 5개 (LB 0.3792)', V2_oof, V2_te), ('32: 시드 10개', S10_oof, S10_te)]:
    sc, pred = fs(po), pt.argmax(1)
    rows.append({'조합': name, 'CV': round(sc.mean(), 4), 'std': round(sc.std(), 4), 'OOF': round(macro_f1(y, po.argmax(1)), 4),
                 'test COAD': f'{np.mean(pred == coad):.1%}', 'test STES': f'{np.mean(pred == stes):.1%}',
                 'APC 절단 → COAD': f'{np.mean(pred[apc_te] == coad):.0%}'})
display(pd.DataFrame(rows))
print(f'v2 재현: CV {s5.mean():.4f} / OOF {macro_f1(y, V2_oof.argmax(1)):.4f} (기록 0.4641 / 0.4651)')
print(f'시드 10개 − v2: {(s10 - s5).mean():+.4f}, {(s10 > s5).sum()}/5 fold | fold별 {(s10 - s5).round(4)}')
print(f'v2와 test 예측이 다른 환자 {np.mean(S10_te.argmax(1) != V2_te.argmax(1)):.1%}')

,조합,CV,std,OOF,test COAD,test STES,APC 절단 → COAD
0,v2: 시드 5개 (LB 0.3792),0.4641,0.0083,0.4651,17.6%,18.7%,67%
1,32: 시드 10개,0.4650,0.0087,0.4659,17.2%,19.0%,66%


v2 재현: CV 0.4641 / OOF 0.4651 (기록 0.4641 / 0.4651)
시드 10개 − v2: +0.0009, 4/5 fold | fold별 [ 0.0002  0.0011  0.0025 -0.0002  0.0012]
v2와 test 예측이 다른 환자 1.3%


#### 참고: 섞기 비율 확인 (train OOF만, 관찰용)
- dedupe를 끄면서 XGB 예측이 바뀌었으니 0.7이 여전히 괜찮은지만 봄
- 0.7과 차이가 std보다 작으면 **0.7 유지** (LB를 보고 비율을 고르지 않음)

In [ ]:
base7 = fs(blend(SEEDS, 0.7)[0])
rows = []
for w in [0.6, 0.7, 0.8]:
    sc = fs(blend(SEEDS, w)[0])
    rows.append({'XGB 비율': w, 'CV': round(sc.mean(), 4), 'std': round(sc.std(), 4),
                 '0.7 대비': round((sc - base7).mean(), 4), '0.7보다 높은 fold': f'{int((sc > base7).sum())}/5'})
pd.DataFrame(rows)

,XGB 비율,CV,std,0.7 대비,0.7보다 높은 fold
0,0.6,0.4667,0.0062,0.0017,2/5
1,0.7,0.4650,0.0087,0.0000,0/5
2,0.8,0.4621,0.0086,-0.0029,0/5


#### 제출 파일 저장 (Windows에서만, 섞기 비율 0.7)

In [ ]:
ok = abs(s5.mean() - 0.4641) < 0.002
if not ok:
    print('⚠️ v2 재현 실패 → 저장하지 않음. 데이터·features.py 설정 확인')
elif platform.system() != 'Windows':
    print(f'{platform.system()}에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)')
else:
    save_submission(submission, S10_te, le, name='blend_ratio22sub100_seed10_0.7_LR0.3_nodedupe',
                    cv_score=macro_f1(y, S10_oof.argmax(1)))

저장 완료: C:\Users\USER\Desktop\hackathon\submissions\1113_blend_ratio22sub100_seed10_0.7_LR0.3_nodedupe_0.466.csv


In [ ]:
# v2 제출 파일(LB 0.3792)과 예측 일치율
old = sorted(glob.glob('../submissions/*seed5_0.7_LR0.3_nodedupe*.csv'))
new = sorted(glob.glob('../submissions/*seed10_0.7_LR0.3_nodedupe*.csv'))
if old and new:
    a, b = pd.read_csv(old[-1]), pd.read_csv(new[-1])
    print(Path(old[-1]).name, 'vs', Path(new[-1]).name)
    print('예측 일치율:', round((a.iloc[:, 1] == b.iloc[:, 1]).mean(), 4))
else:
    print('비교할 파일 없음:', old, new)

1113_blend_ratio22sub100_seed5_0.7_LR0.3_nodedupe_0.465.csv vs 1113_blend_ratio22sub100_seed10_0.7_LR0.3_nodedupe_0.466.csv
예측 일치율: 0.9866


| 32: 시드 10개 | 0.4650 ± 0.0087 | 0.4659 | 4/5 | 1.3% | (기입) |

- 시드별 22b CV 범위: 0.4475 ~ 0.4580 (차이 0.0106) → 시드 하나로는 흔들림이 커서 평균이 필요
- 섞기 비율 확인: 0.6이 +0.0017이지만 2/5 fold, std보다 작음 → 0.7 유지

## 결론
- v2 재현 확인 (CV 0.4641 / OOF 0.4651, 기록과 같음)
- 시드 10개: CV +0.0009 (4/5 fold), test 예측은 v2와 1.3%만 다름 → 같은 모델에서 흔들림만 줄인 버전
- LB: 0.3772 -> 최종 모델은 기존 그대로 v2